In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import NearestNeighbors

eps = 1e-6
k = 3

# =========================
# 1. Load training datasets
# =========================
water_quality = pd.read_csv("../../data/water_quality_training_dataset.csv")
landsat = pd.read_csv("../../data/landsat_features_training.csv")
terraclimate = pd.read_csv("../../data/terraclimate_features_training.csv")

# =========================
# 2. Convert dates
# =========================
water_quality["Sample Date"] = pd.to_datetime(water_quality["Sample Date"], dayfirst=True)
landsat["Sample Date"] = pd.to_datetime(landsat["Sample Date"], dayfirst=True)
terraclimate["Sample Date"] = pd.to_datetime(terraclimate["Sample Date"], dayfirst=True)

# =========================
# 3. Create temporal features
# =========================
water_quality["month"] = water_quality["Sample Date"].dt.month
water_quality["year"] = water_quality["Sample Date"].dt.year
water_quality["dayofyear"] = water_quality["Sample Date"].dt.dayofyear

# =========================
# 4. Merge training datasets
# =========================
df = water_quality.merge(
    landsat,
    on=["Latitude", "Longitude", "Sample Date"],
    how="left"
)

df = df.merge(
    terraclimate,
    on=["Latitude", "Longitude", "Sample Date"],
    how="left"
)

# =========================
# 5. Feature engineering (same core as v4.0)
# =========================
df["nir_swir16_ratio"] = df["nir"] / (df["swir16"] + eps)
df["nir_swir22_ratio"] = df["nir"] / (df["swir22"] + eps)
df["green_nir_ratio"] = df["green"] / (df["nir"] + eps)

df["nir_minus_swir16"] = df["nir"] - df["swir16"]
df["nir_minus_green"] = df["nir"] - df["green"]

df["ndmi_pet"] = df["NDMI"] * df["pet"]
df["mndwi_pet"] = df["MNDWI"] * df["pet"]

df["swir_ratio"] = df["swir16"] / (df["swir22"] + eps)

df["nir_pet"] = df["nir"] * df["pet"]
df["swir16_pet"] = df["swir16"] * df["pet"]
df["ndmi_day"] = df["NDMI"] * df["dayofyear"]

df.replace([np.inf, -np.inf], np.nan, inplace=True)

# =========================
# 6. Handle missing values
# =========================
train_medians = df.median(numeric_only=True)
df.fillna(train_medians, inplace=True)

# =========================
# 7. Spatial neighbor features for TRAIN
#    Use unique training stations and target means
# =========================
targets = [
    "Total Alkalinity",
    "Electrical Conductance",
    "Dissolved Reactive Phosphorus"
]

station_targets = (
    df.groupby(["Latitude", "Longitude"], as_index=False)[targets]
    .mean()
    .rename(columns={
        "Total Alkalinity": "station_alk_mean",
        "Electrical Conductance": "station_ec_mean",
        "Dissolved Reactive Phosphorus": "station_drp_mean"
    })
)

station_coords = station_targets[["Latitude", "Longitude"]].values
station_target_values = station_targets[
    ["station_alk_mean", "station_ec_mean", "station_drp_mean"]
].values

nn_train = NearestNeighbors(
    n_neighbors=min(k + 1, len(station_targets)),
    metric="euclidean"
)
nn_train.fit(station_coords)

row_coords = df[["Latitude", "Longitude"]].values
distances_train, indices_train = nn_train.kneighbors(row_coords)

clean_distances = []
clean_indices = []

for d_row, i_row in zip(distances_train, indices_train):
    mask_nonzero = d_row > 0
    d_filtered = d_row[mask_nonzero]
    i_filtered = i_row[mask_nonzero]

    if len(i_filtered) < k:
        d_filtered = d_row[:k]
        i_filtered = i_row[:k]
    else:
        d_filtered = d_filtered[:k]
        i_filtered = i_filtered[:k]

    clean_distances.append(d_filtered)
    clean_indices.append(i_filtered)

clean_distances = np.array(clean_distances)
clean_indices = np.array(clean_indices)

neighbor_targets_train = station_target_values[clean_indices]

df["knn_dist_mean"] = clean_distances.mean(axis=1)
df["knn_dist_min"] = clean_distances.min(axis=1)

df["knn_alk_mean"] = neighbor_targets_train[:, :, 0].mean(axis=1)
df["knn_ec_mean"] = neighbor_targets_train[:, :, 1].mean(axis=1)
df["knn_drp_mean"] = neighbor_targets_train[:, :, 2].mean(axis=1)

# =========================
# 8. Define training features and targets
# =========================
X = df.drop(columns=targets + ["Sample Date", "Latitude", "Longitude"])
y = df[targets]

X_train_medians = X.median(numeric_only=True)

# =========================
# 9. Train final model
# =========================
rf_final = RandomForestRegressor(
    n_estimators=500,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

rf_final.fit(X, y)

# =========================
# 10. Load submission datasets
# =========================
submission = pd.read_csv("../../data/submission_template.csv")
landsat_val = pd.read_csv("../../data/landsat_features_validation.csv")
terraclimate_val = pd.read_csv("../../data/terraclimate_features_validation.csv")

# =========================
# 11. Convert dates
# =========================
submission["Sample Date"] = pd.to_datetime(submission["Sample Date"], dayfirst=True)
landsat_val["Sample Date"] = pd.to_datetime(landsat_val["Sample Date"], dayfirst=True)
terraclimate_val["Sample Date"] = pd.to_datetime(terraclimate_val["Sample Date"], dayfirst=True)

# =========================
# 12. Create temporal features
# =========================
submission["month"] = submission["Sample Date"].dt.month
submission["year"] = submission["Sample Date"].dt.year
submission["dayofyear"] = submission["Sample Date"].dt.dayofyear

# =========================
# 13. Merge validation datasets
# =========================
df_val = submission.merge(
    landsat_val,
    on=["Latitude", "Longitude", "Sample Date"],
    how="left"
)

df_val = df_val.merge(
    terraclimate_val,
    on=["Latitude", "Longitude", "Sample Date"],
    how="left"
)

# =========================
# 14. Same core feature engineering
# =========================
df_val["nir_swir16_ratio"] = df_val["nir"] / (df_val["swir16"] + eps)
df_val["nir_swir22_ratio"] = df_val["nir"] / (df_val["swir22"] + eps)
df_val["green_nir_ratio"] = df_val["green"] / (df_val["nir"] + eps)

df_val["nir_minus_swir16"] = df_val["nir"] - df_val["swir16"]
df_val["nir_minus_green"] = df_val["nir"] - df_val["green"]

df_val["ndmi_pet"] = df_val["NDMI"] * df_val["pet"]
df_val["mndwi_pet"] = df_val["MNDWI"] * df_val["pet"]

df_val["swir_ratio"] = df_val["swir16"] / (df_val["swir22"] + eps)

df_val["nir_pet"] = df_val["nir"] * df_val["pet"]
df_val["swir16_pet"] = df_val["swir16"] * df_val["pet"]
df_val["ndmi_day"] = df_val["NDMI"] * df_val["dayofyear"]

df_val.replace([np.inf, -np.inf], np.nan, inplace=True)

# =========================
# 15. Spatial neighbor features for VALIDATION
#     Use only training stations
# =========================
coords_val = df_val[["Latitude", "Longitude"]].values

nn_val = NearestNeighbors(
    n_neighbors=min(k, len(station_targets)),
    metric="euclidean"
)
nn_val.fit(station_coords)

distances_val, indices_val = nn_val.kneighbors(coords_val)
neighbor_targets_val = station_target_values[indices_val]

df_val["knn_dist_mean"] = distances_val.mean(axis=1)
df_val["knn_dist_min"] = distances_val.min(axis=1)

df_val["knn_alk_mean"] = neighbor_targets_val[:, :, 0].mean(axis=1)
df_val["knn_ec_mean"] = neighbor_targets_val[:, :, 1].mean(axis=1)
df_val["knn_drp_mean"] = neighbor_targets_val[:, :, 2].mean(axis=1)

# =========================
# 16. Prepare validation features
# =========================
X_val = df_val.drop(
    columns=[
        "Sample Date",
        "Latitude",
        "Longitude",
        "Total Alkalinity",
        "Electrical Conductance",
        "Dissolved Reactive Phosphorus"
    ],
    errors="ignore"
)

X_val = X_val.reindex(columns=X.columns)
X_val = X_val.fillna(X_train_medians)

# =========================
# 17. Predict
# =========================
predictions = rf_final.predict(X_val)

# =========================
# 18. Build submission
# =========================
submission["Total Alkalinity"] = predictions[:, 0]
submission["Electrical Conductance"] = predictions[:, 1]
submission["Dissolved Reactive Phosphorus"] = predictions[:, 2]

submission_v4_3 = submission[
    [
        "Longitude",
        "Latitude",
        "Sample Date",
        "Total Alkalinity",
        "Electrical Conductance",
        "Dissolved Reactive Phosphorus"
    ]
]

# =========================
# 19. Export
# =========================
submission_v4_3.to_csv("../../submissions/submission_v4.3.csv", index=False)

# =========================
# 20. Quick check
# =========================
print(submission_v4_3.shape)
print(submission_v4_3.head())
print(submission_v4_3.isna().sum())

(200, 6)
   Longitude   Latitude Sample Date  Total Alkalinity  Electrical Conductance  \
0  27.822778 -32.043333  2014-09-01        146.414758              519.207047   
1  26.077500 -33.329167  2015-09-16        182.612358              571.830820   
2  27.640028 -32.991639  2015-05-07        131.819407              453.523080   
3  24.439167 -34.096389  2012-02-07        118.429673              572.826800   
4  28.581667 -32.000556  2014-10-01        126.729514              566.065810   

   Dissolved Reactive Phosphorus  
0                      30.785333  
1                      33.256000  
2                      28.533000  
3                      20.690000  
4                      30.216500  
Longitude                        0
Latitude                         0
Sample Date                      0
Total Alkalinity                 0
Electrical Conductance           0
Dissolved Reactive Phosphorus    0
dtype: int64
